# **Notebook 5: Solution V1 — RAG Evaluation**
## Assignment: Hybrid RAG & Fine-Tuning for Customer Support
---

### TO-DO: Before Running This Notebook

**Files you NEED:**
- [ ] `./chroma_db/` — Created by Notebook 4
- [ ] `df_test.csv` — Created by Notebook 2
- [ ] `outputs.json` — Created by Notebooks 3+4
- [ ] GPU runtime enabled

**Files this notebook will CREATE:**
- [ ] `v1_metrics.csv` — Per-row Baseline vs V1 scores _(Evidence for comparative analysis)_

---

### **Task 3.3: Evaluate Solution V1**

> This task is split into five measurements (3.3.1–3.3.5). Run the shared setup cell below first (it loads the model, ChromaDB, and test data), then work through each measurement.

**── Shared setup ──**
Load the base model, reload ChromaDB (same embedding model as NB4), and load `df_test.csv` + `outputs.json`. Define helper functions `generate_baseline()` and `generate_naive_rag()` here so every subtask below can reuse them.

In [1]:
import json
import numpy as np
import pandas as pd
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"                       # same as every other notebook
EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"    # same as Notebook 4
CHROMA_DIR = "./chroma_db"
DEVICE = "mps" if torch.backends.mps.is_available() else ("cuda" if torch.cuda.is_available() else "cpu")
GEN_KWARGS = dict(max_new_tokens=120, do_sample=False, temperature=None, top_p=None)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
base_model = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=torch.float16).to(DEVICE)
base_model.eval()

embeddings = HuggingFaceEmbeddings(model_name=EMBEDDING_MODEL)
vector_db = Chroma(persist_directory=CHROMA_DIR, embedding_function=embeddings)
print("ChromaDB reloaded:", vector_db._collection.count(), "documents")

df_test = pd.read_csv("df_test.csv")
with open("outputs.json") as f:
    outputs = json.load(f)
print(f"df_test shape: {df_test.shape}")

BASELINE_SYSTEM_PROMPT = "You are a helpful customer support assistant."
RAG_SYSTEM_TEMPLATE = (
    "You are a customer support assistant. Answer strictly using this SOP:\n\n{context}\n\n"
    "If the SOP does not cover the question, say you will escalate to a human agent."
)

def generate_baseline(query):
    messages = [{"role": "system", "content": BASELINE_SYSTEM_PROMPT}, {"role": "user", "content": query}]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        out = base_model.generate(**inputs, pad_token_id=tokenizer.pad_token_id, **GEN_KWARGS)
    return tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

def generate_naive_rag(query, k=1):
    docs = vector_db.similarity_search(query, k=k)
    context = "\n\n".join(d.page_content for d in docs)
    system_prompt = RAG_SYSTEM_TEMPLATE.format(context=context)
    messages = [{"role": "system", "content": system_prompt}, {"role": "user", "content": query}]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        out = base_model.generate(**inputs, pad_token_id=tokenizer.pad_token_id, **GEN_KWARGS)
    answer = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()
    return answer, docs

print("Setup complete: generate_baseline() and generate_naive_rag() are ready for reuse below.")


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

ChromaDB reloaded: 13 documents
df_test shape: (149, 5)
Setup complete: generate_baseline() and generate_naive_rag() are ready for reuse below.


#### **3.3.1 Execute Automated Testing [3 marks]**
**The Task:** Run both Baseline and Naive RAG across the entire held-out test set, collecting their generated outputs for every row.

**Hints & Tips:**
* Loop over `df_test` rows; for each query call both `generate_baseline()` and `generate_naive_rag()`.
* Store raw outputs in a list of dicts so the later measurements can score them.
* This is the most time-consuming cell — if constrained, `df_test.sample(50)` is acceptable.

**Learner Inference:** Automated testing across the full set gives statistically meaningful results, not a single cherry-picked query.

In [2]:
# Full held-out test set has ~149 rows. Running TWO LLM generation passes per row (baseline +
# naive RAG) on local CPU/MPS hardware (no T4 GPU) is by far the most time-consuming step in this
# project. Per the assignment's own guidance ("if constrained, df_test.sample(50) is acceptable"),
# we evaluate a random subsample of the held-out test split, which keeps runtime tractable while
# remaining a statistically meaningful, leakage-free sample (drawn only from the test partition).
EVAL_SAMPLE_SIZE = 60
eval_df = df_test.sample(min(EVAL_SAMPLE_SIZE, len(df_test)), random_state=42).reset_index(drop=True)
print(f"Evaluating on {len(eval_df)}/{len(df_test)} held-out test rows, "
      f"spanning {eval_df['intent'].nunique()}/{df_test['intent'].nunique()} intents.")

results = []
for i, row in eval_df.iterrows():
    query = row["instruction"]
    baseline_out = generate_baseline(query)
    rag_out, docs = generate_naive_rag(query, k=1)
    results.append({
        "instruction": query,
        "intent": row["intent"],
        "category": row["category"],
        "baseline_output": baseline_out,
        "naive_rag_output": rag_out,
        "retrieved_doc": docs[0].metadata["source_file"],
    })
    if (i + 1) % 10 == 0:
        print(f"  processed {i + 1}/{len(eval_df)}")

results_df = pd.DataFrame(results)
print(f"\nCompleted automated testing on {len(results_df)} rows.")
results_df.head(3)


Evaluating on 60/149 held-out test rows, spanning 24/27 intents.


  processed 10/60


  processed 20/60


  processed 30/60


  processed 40/60


  processed 50/60


  processed 60/60

Completed automated testing on 60 rows.


,instruction,intent,category,baseline_output,naive_rag_output,retrieved_doc
0,"I want to see how soon can I expect the order,...",delivery_period,DELIVERY,To check the estimated delivery date for your ...,To check the expected delivery time for your o...,shipping_delays.md
1,i have lost my account access key where can i...,recover_password,ACCOUNT,I'm sorry to hear that you've lost your access...,I'm sorry to hear that you've lost your accoun...,account_recovery.md
2,I need help closing a platinum account,delete_account,ACCOUNT,"Sure, I'd be happy to assist you with that! Cl...","I'm sorry, but I can't assist with that request.",account_recovery.md


#### **3.3.2 Measure Format Adherence [2 marks]**
**The Task:** Validate the syntactic correctness of the generated outputs and report the adherence rate.

**Hints & Tips:**
* For the baseline/RAG free-text responses, "format adherence" means the output is well-formed and non-empty (the strict JSON check applies mainly to the fine-tuned router in NB7).
* Report the percentage of outputs that parsed/validated successfully.

**Learner Inference:** Format adherence tells you how often the system produces usable output before you even check correctness.

In [3]:
def is_well_formed(text):
    """For free-text Baseline/RAG responses, 'format adherence' = a non-empty, non-truncated answer."""
    if not isinstance(text, str):
        return False
    t = text.strip()
    return len(t) > 0 and not t.endswith((",", ":", "and", "the", "a"))

results_df["baseline_well_formed"] = results_df["baseline_output"].apply(is_well_formed)
results_df["rag_well_formed"] = results_df["naive_rag_output"].apply(is_well_formed)

baseline_adherence = results_df["baseline_well_formed"].mean() * 100
rag_adherence = results_df["rag_well_formed"].mean() * 100

print(f"Baseline Format Adherence Rate: {baseline_adherence:.1f}%")
print(f"Naive RAG Format Adherence Rate: {rag_adherence:.1f}%")


Baseline Format Adherence Rate: 91.7%
Naive RAG Format Adherence Rate: 98.3%


#### **3.3.3 Measure Execution Success (ROUGE/BLEU) [2 marks]**
**The Task:** Evaluate semantic similarity of each output against SOP-grounded references using ROUGE-1, ROUGE-L, and BLEU.

**Hints & Tips:**
* Use SOP-grounded references — retrieve the correct SOP per test row so policy-specific language is rewarded.
* Generic references falsely reward vague baseline answers — avoid them.
* `rouge_scorer.RougeScorer(['rouge1','rougeL'], use_stemmer=True)` and `sentence_bleu` with `SmoothingFunction().method1`.

**Learner Inference:** ROUGE/BLEU measure how close the output is to a correct, policy-grounded answer.

In [4]:
from rouge_score import rouge_scorer
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

scorer = rouge_scorer.RougeScorer(["rouge1", "rougeL"], use_stemmer=True)
smoothie = SmoothingFunction().method1

# SOP-grounded references: for each row's TRUE intent/category, retrieve the matching SOP
# document via the vector index (semantic match on the clean label, not the noisy raw query).
# Generic references would falsely reward vague baseline answers, so we deliberately ground
# every reference in the actual company policy text.
def get_sop_reference(intent, category):
    query = f"{intent} {category}".replace("_", " ")
    docs = vector_db.similarity_search(query, k=1)
    return docs[0].page_content, docs[0].metadata["source_file"]

ref_cache = {}
for _, r in results_df[["intent", "category"]].drop_duplicates().iterrows():
    ref_cache[(r["intent"], r["category"])] = get_sop_reference(r["intent"], r["category"])

def score_row(generated, intent, category):
    ref_text, ref_file = ref_cache[(intent, category)]
    rouge = scorer.score(ref_text, generated)
    bleu = sentence_bleu([ref_text.split()], generated.split(), smoothing_function=smoothie)
    return rouge["rouge1"].fmeasure, rouge["rougeL"].fmeasure, bleu, ref_file

baseline_scores = results_df.apply(lambda r: score_row(r["baseline_output"], r["intent"], r["category"]), axis=1)
rag_scores = results_df.apply(lambda r: score_row(r["naive_rag_output"], r["intent"], r["category"]), axis=1)

(results_df["baseline_rouge1"], results_df["baseline_rougeL"],
 results_df["baseline_bleu"], results_df["sop_reference_file"]) = zip(*baseline_scores)
(results_df["rag_rouge1"], results_df["rag_rougeL"],
 results_df["rag_bleu"], _) = zip(*rag_scores)

print("Mean ROUGE-1 — Baseline: {:.3f} | Naive RAG: {:.3f}".format(
    results_df["baseline_rouge1"].mean(), results_df["rag_rouge1"].mean()))
print("Mean ROUGE-L — Baseline: {:.3f} | Naive RAG: {:.3f}".format(
    results_df["baseline_rougeL"].mean(), results_df["rag_rougeL"].mean()))
print("Mean BLEU    — Baseline: {:.3f} | Naive RAG: {:.3f}".format(
    results_df["baseline_bleu"].mean(), results_df["rag_bleu"].mean()))


Mean ROUGE-1 — Baseline: 0.158 | Naive RAG: 0.111
Mean ROUGE-L — Baseline: 0.081 | Naive RAG: 0.073
Mean BLEU    — Baseline: 0.001 | Naive RAG: 0.001


#### **3.3.4 Measure Output Consistency [1 mark]**
**The Task:** Evaluate deterministic behaviour by running the same query multiple times under `do_sample=False` and confirming identical outputs.

**Hints & Tips:**
* Run the same query 3 times; assert all outputs are identical.
* With `do_sample=False, temperature=None`, greedy decoding should be fully deterministic.

**Learner Inference:** Deterministic inference means your evaluation is reproducible — the same input always gives the same output.

In [5]:
consistency_query = eval_df.iloc[0]["instruction"]

repeat_baseline = [generate_baseline(consistency_query) for _ in range(3)]
all_identical_baseline = len(set(repeat_baseline)) == 1
print("Repeated Baseline outputs identical (determinism check):", all_identical_baseline)

repeat_rag = [generate_naive_rag(consistency_query, k=1)[0] for _ in range(3)]
all_identical_rag = len(set(repeat_rag)) == 1
print("Repeated Naive RAG outputs identical:", all_identical_rag)

consistency_rate_baseline = 100.0 if all_identical_baseline else 0.0
consistency_rate_rag = 100.0 if all_identical_rag else 0.0
print(f"\nBaseline Consistency Rate: {consistency_rate_baseline:.0f}%")
print(f"Naive RAG Consistency Rate: {consistency_rate_rag:.0f}%")
print("\n(Both use do_sample=False, temperature=None — greedy decoding is deterministic by construction.)")


Repeated Baseline outputs identical (determinism check): True


Repeated Naive RAG outputs identical: True

Baseline Consistency Rate: 100%
Naive RAG Consistency Rate: 100%

(Both use do_sample=False, temperature=None — greedy decoding is deterministic by construction.)


#### **3.3.5 Measure Hallucination Frequency [2 marks]**
**The Task:** Evaluate how often outputs contain unsupported claims, invalid references, missing functionality, or policy violations.

**Hints & Tips:**
* Compare outputs against the retrieved SOP — flag any specific claim (dates, numbers, policies) not supported by the context.
* Report hallucination frequency as a percentage for both Baseline and Naive RAG.

**Learner Inference:** This quantifies the core problem RAG is meant to solve — grounding responses to reduce fabrication.

In [6]:
import re

UNGROUNDED_TERMS = [
    "tracking number", "tracking id", "click here", "our app", "download the app",
    "website", "24/7 hotline", "live chat", "call center",
]

def contains_unsupported_numbers(generated, reference):
    """Flag numeric claims (dates, day-counts, etc.) in the output not present in the SOP reference."""
    gen_nums = set(re.findall(r"\d+", generated))
    ref_nums = set(re.findall(r"\d+", reference))
    return len(gen_nums - ref_nums) > 0

def has_ungrounded_terms(generated, reference):
    gen_low, ref_low = generated.lower(), reference.lower()
    return any(term in gen_low and term not in ref_low for term in UNGROUNDED_TERMS)

def is_hallucinated(generated, intent, category):
    ref_text, _ = ref_cache[(intent, category)]
    return contains_unsupported_numbers(generated, ref_text) or has_ungrounded_terms(generated, ref_text)

results_df["baseline_hallucinated"] = results_df.apply(
    lambda r: is_hallucinated(r["baseline_output"], r["intent"], r["category"]), axis=1)
results_df["rag_hallucinated"] = results_df.apply(
    lambda r: is_hallucinated(r["naive_rag_output"], r["intent"], r["category"]), axis=1)

baseline_halluc_rate = results_df["baseline_hallucinated"].mean() * 100
rag_halluc_rate = results_df["rag_hallucinated"].mean() * 100

print(f"Baseline Hallucination Frequency: {baseline_halluc_rate:.1f}%")
print(f"Naive RAG Hallucination Frequency: {rag_halluc_rate:.1f}%")
print("\n(Flags unsupported numeric claims and ungrounded references to mechanisms/URLs "
      "absent from the SOP-grounded reference for that row's true intent.)")


Baseline Hallucination Frequency: 25.0%
Naive RAG Hallucination Frequency: 23.3%

(Flags unsupported numeric claims and ungrounded references to mechanisms/URLs absent from the SOP-grounded reference for that row's true intent.)


### **Task 3.4: Analyse Retrieval Impact**

#### **3.4.1 Compare Baseline and Solution V1 [4 marks]**
**The Task:** Quantify the impact of retrieval by comparing aggregate scores across Functional Correctness, Consistency, and Hallucination Frequency, with percentage changes.

**Hints & Tips:**
* Build a summary table: Baseline vs Naive RAG for each metric.
* Compute improvement percentages: `(rag - base) / base * 100`.
* Document WHERE retrieval helps and where it doesn't — both motivate Stage 4.

**Learner Inference:** This isolates retrieval's independent contribution before fine-tuning enters the picture.

In [7]:
def pct_change(base, new):
    if base == 0:
        return np.nan
    return (new - base) / base * 100

summary = pd.DataFrame({
    "Metric": ["Format Adherence (%)", "ROUGE-1", "ROUGE-L", "BLEU",
               "Consistency Rate (%)", "Hallucination Frequency (%)"],
    "Baseline": [
        baseline_adherence, results_df["baseline_rouge1"].mean(), results_df["baseline_rougeL"].mean(),
        results_df["baseline_bleu"].mean(), consistency_rate_baseline, baseline_halluc_rate,
    ],
    "Naive RAG (V1)": [
        rag_adherence, results_df["rag_rouge1"].mean(), results_df["rag_rougeL"].mean(),
        results_df["rag_bleu"].mean(), consistency_rate_rag, rag_halluc_rate,
    ],
})
summary["Change (%)"] = summary.apply(lambda r: pct_change(r["Baseline"], r["Naive RAG (V1)"]), axis=1)

print(summary.to_string(index=False))
print("\nNote: for every metric EXCEPT Hallucination Frequency, positive Change(%) = improvement.")
print("For Hallucination Frequency, NEGATIVE Change(%) = improvement (fewer hallucinations).")

halluc_change = summary.loc[summary["Metric"] == "Hallucination Frequency (%)", "Change (%)"].iloc[0]
rouge1_change = summary.loc[summary["Metric"] == "ROUGE-1", "Change (%)"].iloc[0]
print(f"\nRetrieval's independent impact: hallucination frequency changed by {halluc_change:+.1f}% "
      f"relative to baseline (negative = improvement); ROUGE-1 moved by {rouge1_change:+.1f}%.")


                     Metric   Baseline  Naive RAG (V1)  Change (%)
       Format Adherence (%)  91.666667       98.333333    7.272727
                    ROUGE-1   0.157849        0.111389  -29.433234
                    ROUGE-L   0.081398        0.073041  -10.267146
                       BLEU   0.001047        0.000928  -11.347992
       Consistency Rate (%) 100.000000      100.000000    0.000000
Hallucination Frequency (%)  25.000000       23.333333   -6.666667

Note: for every metric EXCEPT Hallucination Frequency, positive Change(%) = improvement.
For Hallucination Frequency, NEGATIVE Change(%) = improvement (fewer hallucinations).

Retrieval's independent impact: hallucination frequency changed by -6.7% relative to baseline (negative = improvement); ROUGE-1 moved by -29.4%.


---
## Save Artifacts

In [8]:
results_df.to_csv("v1_metrics.csv", index=False)
summary.to_csv("v1_summary_metrics.csv", index=False)
print("Saved v1_metrics.csv (per-row) and v1_summary_metrics.csv (aggregate).")


Saved v1_metrics.csv (per-row) and v1_summary_metrics.csv (aggregate).


---
## END-OF-NOTEBOOK CHECKLIST

> **IMPORTANT: Verify before proceeding to Notebook 6.**

- [ ] ChromaDB reloaded from `./chroma_db/`
- [ ] **3.3.1** Automated testing run across full test set
- [ ] **3.3.2** Format adherence measured
- [ ] **3.3.3** ROUGE/BLEU computed with SOP-grounded references
- [ ] **3.3.4** Output consistency (determinism) verified
- [ ] **3.3.5** Hallucination frequency quantified
- [ ] **3.4.1** Retrieval impact quantified with improvement %
- [ ] **`v1_metrics.csv` saved** ← _Evidence for comparative analysis_

**If any item is unchecked, fix it before moving on.**